[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap08_machine_learning/exemplo_classificacao_pedidos_prazo.ipynb)

# Classificação de pedidos segundo cumprimento do prazo

Este notebook compara regressão logística e árvore de decisão para classificar pedidos da Base didática Caso Tech conforme o cumprimento do prazo de entrega. O protocolo de avaliação é comum aos modelos. O alvo e os preditores devem ser conferidos antes do ajuste, com exclusão explícita de informações disponíveis somente após a entrega.

O botão pressupõe que o arquivo seja publicado no caminho `capitulos/cap08_machine_learning/exemplo_classificacao_pedidos_prazo.ipynb` do repositório. Ajuste o endereço caso a pasta definida para o capítulo seja outra.

## 1. Objetivo e enquadramento

O problema é de aprendizado supervisionado e classificação binária: cada pedido recebe um rótulo que indica se a entrega cumpriu o prazo. O modelo aprende a partir de pedidos cujo resultado já é conhecido e estima a classe para novos casos.

A definição operacional do alvo precisa ser validada com a estrutura real da base. O notebook procura uma coluna de status de cumprimento ou, se necessário, datas de prazo previsto e entrega realizada. Não transforma automaticamente o prazo numérico em classes sem explicitar a regra. A unidade de análise é um pedido, e a duplicidade de registros por pedido deve ser examinada antes da modelagem.

In [ ]:
import io
import re
import unicodedata
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score,
                             roc_auc_score)

SEMENTE = 2026
URL_BASE = "https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/base%20did%C3%A1tica%20Caso%20Tech.xlsx"
np.random.seed(SEMENTE)
print("Semente:", SEMENTE)

## 2. Carregamento e inspeção

A base é carregada do repositório indicado. Caso o download não funcione no Google Colab, o notebook solicita o envio local da planilha. São exibidos os nomes das abas, dimensões e colunas para permitir a conferência da estrutura antes de definir o alvo e os atributos.

In [ ]:
def carregar_excel():
    try:
        with urllib.request.urlopen(URL_BASE, timeout=30) as resposta:
            conteudo = resposta.read()
        excel = pd.ExcelFile(io.BytesIO(conteudo))
        origem = "repositório GitHub"
    except Exception as erro:
        print("Download automático indisponível:", erro)
        try:
            from google.colab import files
            enviados = files.upload()
            if not enviados:
                raise RuntimeError("Nenhum arquivo selecionado.")
            nome = next(iter(enviados))
            excel = pd.ExcelFile(io.BytesIO(enviados[nome]))
            origem = nome
        except ImportError:
            raise RuntimeError("Execute no Colab ou carregue a planilha no ambiente.")
    return excel, origem

excel, origem = carregar_excel()
abas = {aba: pd.read_excel(excel, sheet_name=aba) for aba in excel.sheet_names}
print("Origem:", origem)
for aba, tabela in abas.items():
    print(f"\n{aba}: {tabela.shape[0]} linhas x {tabela.shape[1]} colunas")
    print(list(tabela.columns))

## 3. Definição e validação do alvo

O alvo deve indicar cumprimento do prazo na unidade de pedido. Quando houver uma coluna categórica de status, suas categorias precisam ser inspecionadas e mapeadas explicitamente para “no prazo” e “fora do prazo”. Como alternativa, datas previstas e realizadas podem definir o resultado, desde que a regra de negócio seja confirmada. A data realizada pode ser usada para criar o rótulo, mas nunca como preditor, pois é conhecida somente depois da entrega.

In [ ]:
def normalizar(x):
    x = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", x.lower()).strip()

def encontrar_coluna(colunas, requisitos):
    for req in requisitos:
        for c in colunas:
            nome = normalizar(c)
            if all(p in nome for p in req):
                return c
    return None

# Busca a aba com chave de pedido e indicador categórico de prazo ou datas relevantes.
config = None
for aba, df in abas.items():
    chave = encontrar_coluna(df.columns, [["pedido"], ["order", "id"], ["id", "pedido"]])
    status = encontrar_coluna(df.columns, [["cumprimento", "prazo"], ["status", "prazo"], ["entrega", "prazo"]])
    prevista = encontrar_coluna(df.columns, [["data", "prevista"], ["prazo", "entrega"], ["data", "prometida"]])
    realizada = encontrar_coluna(df.columns, [["data", "entrega", "real"], ["entrega", "realizada"], ["data", "recebimento"]])
    if chave and (status or (prevista and realizada)):
        config = (aba, df, chave, status, prevista, realizada)
        break

if config:
    aba_alvo, df_alvo, coluna_pedido, coluna_status, coluna_prevista, coluna_realizada = config
    print("Aba candidata:", aba_alvo)
    print("Chave:", coluna_pedido)
    print("Status candidato:", coluna_status)
    print("Data prevista candidata:", coluna_prevista)
    print("Data realizada candidata:", coluna_realizada)
    if coluna_status:
        display(df_alvo[coluna_status].value_counts(dropna=False).rename("frequência").to_frame())
else:
    print("Detecção automática não encontrou estrutura suficiente. Use as colunas listadas acima e configure a célula seguinte.")

### Configuração manual e regra de rotulagem

Confirme as categorias e a regra operacional. Se usar uma coluna de status, preencha `mapa_status` com as categorias reais observadas e os rótulos `No prazo` ou `Fora do prazo`. Se usar datas, o notebook usa `realizada <= prevista` como regra, que só deve ser mantida se corresponder à definição adotada no caso. Não prossiga com mapeamento presumido.

In [ ]:
if config:
    pass
else:
    aba_alvo = None
    coluna_pedido = None
    coluna_status = None
    coluna_prevista = None
    coluna_realizada = None
    if not aba_alvo:
        raise ValueError("Preencha os nomes exatos da aba e das colunas do alvo conforme a inspeção.")
    df_alvo = abas[aba_alvo]

# Para alvo baseado em status, escreva o mapeamento validado após observar value_counts.
# Exemplo meramente estrutural: {"categoria observada A": "No prazo", "categoria observada B": "Fora do prazo"}
mapa_status = None

if coluna_status:
    if mapa_status is None:
        raise ValueError("Defina mapa_status com as categorias reais da coluna de status.")
    rotulos = df_alvo[coluna_status].astype(str).map(mapa_status)
    alvo = rotulos.rename("cumprimento_prazo")
elif coluna_prevista and coluna_realizada:
    dados_datas = df_alvo[[coluna_prevista, coluna_realizada]].copy()
    dados_datas[coluna_prevista] = pd.to_datetime(dados_datas[coluna_prevista], errors="coerce")
    dados_datas[coluna_realizada] = pd.to_datetime(dados_datas[coluna_realizada], errors="coerce")
    alvo = pd.Series(np.where(dados_datas[coluna_realizada] <= dados_datas[coluna_prevista],
                              "No prazo", "Fora do prazo"), index=df_alvo.index,
                     name="cumprimento_prazo")
    alvo[dados_datas[[coluna_prevista, coluna_realizada]].isna().any(axis=1)] = np.nan
else:
    raise ValueError("Configure uma coluna de status ou datas previstas e realizadas verificadas.")

print("Distribuição do alvo validado:")
display(alvo.value_counts(dropna=False).to_frame("frequência"))

## 4. Granularidade e montagem do conjunto analítico

O objetivo é manter um registro por pedido. Se uma chave aparecer em múltiplas linhas, o notebook verifica se os valores do alvo são consistentes dentro do pedido. Em caso de conflito, o pedido é removido para investigação, em vez de receber um rótulo arbitrário. Para os atributos, consolida-se por primeiro valor não ausente somente após mostrar quais colunas divergem. A regra deve ser ajustada se a estrutura do caso indicar outra forma correta de agregação.

In [ ]:
dados = df_alvo.copy()
dados["__alvo__"] = alvo
if coluna_pedido is None or coluna_pedido not in dados.columns:
    raise ValueError("A chave de pedido precisa ser confirmada e existir na tabela.")

# Exclui rótulos ausentes e mede inconsistências de alvo por pedido.
dados = dados.dropna(subset=[coluna_pedido, "__alvo__"])
rotulos_por_pedido = dados.groupby(coluna_pedido)["__alvo__"].nunique()
pedidos_conflitantes = rotulos_por_pedido[rotulos_por_pedido > 1].index
print("Pedidos com rótulos conflitantes:", len(pedidos_conflitantes))
dados = dados[~dados[coluna_pedido].isin(pedidos_conflitantes)].copy()

# Lista colunas com múltiplos valores por pedido para revisão.
divergencias = []
for c in dados.columns:
    if c in [coluna_pedido, "__alvo__"]:
        continue
    if (dados.groupby(coluna_pedido)[c].nunique(dropna=True) > 1).any():
        divergencias.append(c)
print("Colunas divergentes dentro do pedido:", divergencias)

# Uma linha por pedido: primeiro valor não ausente; revisar divergências antes de interpretar.
pedido = dados.groupby(coluna_pedido, as_index=False).first()
print("Pedidos após consolidação:", len(pedido))

## 5. Seleção dos preditores e prevenção de vazamento

São removidas a chave, as colunas usadas para construir o alvo e variáveis cujos nomes indiquem informação pós-entrega, como data realizada, atraso observado e prazo efetivamente cumprido. Essa filtragem por nome é apenas uma barreira inicial. Cada preditor restante deve ser validado semanticamente: se só puder ser conhecido depois da entrega, deve ser excluído mesmo que seu nome não apareça na lista de termos.

In [ ]:
colunas_alvo = {coluna_pedido, "__alvo__"}
for c in [coluna_status, coluna_prevista, coluna_realizada]:
    if c is not None:
        colunas_alvo.add(c)

termos_pos_entrega = [
    "realizada", "data entrega real", "data recebimento", "atraso",
    "dias atraso", "cumprimento prazo", "status entrega", "entregue em",
    "prazo realizado", "tempo real entrega"
]

preditores = []
excluidas = []
for c in pedido.columns:
    nome = normalizar(c)
    if c in colunas_alvo or any(t in nome for t in termos_pos_entrega):
        excluidas.append(c)
        continue
    taxa_unicos = pedido[c].nunique(dropna=True) / max(len(pedido), 1)
    if taxa_unicos > 0.95:
        excluidas.append(c)
        continue
    preditores.append(c)

print("Colunas excluídas automaticamente:", excluidas)
print("Preditores candidatos:", preditores)
print("IMPORTANTE: revise se cada preditor estava disponível antes do despacho/entrega.")
if not preditores:
    raise ValueError("Nenhum preditor candidato. Revise a estrutura e a disponibilidade temporal.")

# Permite ao autor remover qualquer campo que, após validação semântica, seja pós-evento.
PREDITORES_VALIDADOS = preditores.copy()
# Exemplo: PREDITORES_VALIDADOS.remove("nome exato da coluna a excluir")
if not PREDITORES_VALIDADOS:
    raise ValueError("A lista final de preditores validados está vazia.")

X = pedido[PREDITORES_VALIDADOS].copy()
y = pedido["__alvo__"].astype(str)
print("Classes finais:", sorted(y.unique()))
print("Contagens:")
display(y.value_counts().to_frame("frequência"))

## 6. Protocolo comum de avaliação

Os dois modelos recebem a mesma divisão treino/teste, o mesmo pré-processamento e o mesmo conjunto de preditores. A divisão é estratificada para preservar, tanto quanto possível, a proporção das classes. Se houver dependência temporal ou pedidos agrupados em lotes/clientes, esse protocolo aleatório deverá ser substituído por uma estratégia compatível com a estrutura dos dados.

As métricas incluem acurácia, precisão, revocação e F1. Em problemas de cumprimento de prazo, a classe de interesse e o custo de erros precisam ser definidos antes de escolher o modelo; por isso, nenhuma métrica isolada determina a decisão.

In [ ]:
if y.nunique() != 2:
    raise ValueError("O exercício requer alvo binário. Revise o mapeamento de categorias.")

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=SEMENTE, stratify=y
)

numericas = X_treino.select_dtypes(include=np.number).columns.tolist()
categoricas = [c for c in X_treino.columns if c not in numericas]

prep_num = Pipeline([
    ("imputacao", SimpleImputer(strategy="median")),
    ("escala", StandardScaler())
])
prep_cat = Pipeline([
    ("imputacao", SimpleImputer(strategy="most_frequent")),
    ("codificacao", OneHotEncoder(handle_unknown="ignore"))
])
preprocessador = ColumnTransformer([
    ("num", prep_num, numericas),
    ("cat", prep_cat, categoricas)
])

print("Treino:", len(X_treino), "Teste:", len(X_teste))
print("Classes no treino:", y_treino.value_counts().to_dict())
print("Classes no teste:", y_teste.value_counts().to_dict())

## 7. Ajuste e comparação dos classificadores

A regressão logística é estimada com regularização padrão da biblioteca e limite ampliado de iterações. A árvore de decisão tem profundidade e tamanho mínimo de folha limitados para reduzir complexidade excessiva. Esses parâmetros têm finalidade didática; sua seleção final exigiria validação cruzada dentro do conjunto de treinamento. O conjunto de teste permanece reservado para a avaliação final.

In [ ]:
modelos = {
    "Referência: classe majoritária": DummyClassifier(strategy="most_frequent"),
    "Regressão logística": LogisticRegression(max_iter=2000, class_weight=None),
    "Árvore de decisão": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=10, random_state=SEMENTE, class_weight=None
    )
}

# Classe positiva definida de forma explícita para métricas binárias.
# Ajuste se a categoria de interesse não for exatamente “Fora do prazo”.
classe_positiva = "Fora do prazo"
if classe_positiva not in y.unique():
    raise ValueError("Ajuste classe_positiva às categorias observadas do alvo.")

pipelines = {}
linhas = []
for nome, classificador in modelos.items():
    pipe = Pipeline([
        ("preprocessamento", preprocessador),
        ("classificador", classificador)
    ])
    pipe.fit(X_treino, y_treino)
    pred = pipe.predict(X_teste)
    pipelines[nome] = pipe
    linhas.append({
        "Modelo": nome,
        "Acurácia": accuracy_score(y_teste, pred),
        "Precisão (fora do prazo)": precision_score(y_teste, pred, pos_label=classe_positiva, zero_division=0),
        "Revocação (fora do prazo)": recall_score(y_teste, pred, pos_label=classe_positiva, zero_division=0),
        "F1 (fora do prazo)": f1_score(y_teste, pred, pos_label=classe_positiva, zero_division=0)
    })

comparacao = pd.DataFrame(linhas).set_index("Modelo")
display(comparacao.round(3))

## 8. Matrizes de confusão e interpretação

A matriz de confusão mostra acertos e erros para cada classe. A revocação da classe “Fora do prazo” indica a proporção dos atrasos observados que o modelo identificou; a precisão indica a proporção das previsões de atraso que realmente eram atrasos no conjunto de teste. A escolha entre modelos depende do uso pretendido e das consequências de cada tipo de erro.

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(15, 4))
nomes = list(modelos.keys())
for ax, nome in zip(eixos, nomes):
    pred = pipelines[nome].predict(X_teste)
    cm = confusion_matrix(y_teste, pred, labels=["No prazo", "Fora do prazo"])
    ConfusionMatrixDisplay(cm, display_labels=["No prazo", "Fora do prazo"]).plot(
        ax=ax, colorbar=False, values_format="d"
    )
    ax.set_title(nome)
plt.tight_layout()
plt.show()

for nome in nomes:
    print("\n", nome)
    print(classification_report(y_teste, pipelines[nome].predict(X_teste), zero_division=0))

## 9. Conclusão e limites

A comparação permite examinar dois classificadores no mesmo problema, mantendo constantes a definição do alvo, os preditores, a separação treino/teste e o pré-processamento. A regressão logística oferece uma fronteira de decisão baseada em uma combinação dos atributos transformados; a árvore de decisão organiza regras de partição sucessivas. O desempenho observado depende dos dados e do protocolo, não apenas do algoritmo.

Antes de interpretar os resultados, confirme a regra de cumprimento do prazo, a consistência da unidade pedido, as categorias do alvo e a disponibilidade dos atributos no momento da previsão. Qualquer campo pós-entrega pode produzir avaliação artificialmente otimista. Por fim, um teste aleatório em base didática não garante desempenho futuro nem representa validação operacional.

### Checklist

- Conferir que cada observação corresponde a um pedido.
- Validar o mapeamento do alvo para as duas classes.
- Investigar pedidos com rótulos conflitantes.
- Confirmar temporalmente cada preditor e remover campos pós-entrega.
- Aplicar a mesma divisão e o mesmo pré-processamento aos modelos.
- Comparar métricas por classe e matrizes de confusão.
- Definir a classe de interesse e o custo dos erros antes de escolher o modelo.